# Петухов Кирилл, группа P3123

---

# Лабораторная работа 8. Скрапинг

**Цели:**
1. Спарсить список новостей с сайта https://news.itmo.ru/ru (ID, название, дата, URL)
2. Для каждой новости спарсить подробные данные (ID, название, дата, просмотры, текст, теги)
3. Сохранить все данные в CSV-файлах


## 1. Установка и импорт библиотек


In [ ]:
# Установка необходимых библиотек (в Colab обычно уже установлены)
!pip install requests beautifulsoup4 --quiet


In [ ]:
import csv
import os
import re
import time
import random
import requests
from bs4 import BeautifulSoup

print('Библиотеки успешно загружены.')


## 2. Константы и настройки


In [ ]:
DOMAIN = 'https://news.itmo.ru'
LIST_URL = 'https://news.itmo.ru/ru/main_news/'

# Папка для хранения отдельных файлов новостей
NEWS_CONTENT_DIR = 'news_content'
os.makedirs(NEWS_CONTENT_DIR, exist_ok=True)

# Имя файла с общими данными
SUMMARY_CSV = 'news_summary.csv'

# Задержка между запросами (в секундах), чтобы не перегружать сервер
MIN_DELAY = 0.5
MAX_DELAY = 1.5

# Количество страниц для парсинга (1 страница ~ 10 новостей)
PAGES_TO_PARSE = 5

HEADERS = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) '
                  'AppleWebKit/537.36 (KHTML, like Gecko) '
                  'Chrome/120.0.0.0 Safari/537.36'
}

print(f'Настройки загружены. Будем парсить {PAGES_TO_PARSE} страниц.')
print(f'Папка для новостей: {NEWS_CONTENT_DIR}/')


## 3. Сбор списка новостей (ID, название, дата, URL)

Парсим страницы `/ru/main_news/` и извлекаем основные поля.


In [ ]:
def extract_id_from_url(url):
    """
    Извлекает числовой ID новости из URL.
    Пример: https://news.itmo.ru/ru/education/trend/news/14771/ -> 14771
    """
    match = re.search(r'/(\d+)/?$', url)
    if match:
        return int(match.group(1))
    return None


def parse_list_page(page_num):
    """
    Парсит одну страницу списка новостей.
    Возвращает список словарей с ключами: id, title, date, url.
    """
    url = LIST_URL + str(page_num) + '/'
    results = []

    try:
        resp = requests.get(url, headers=HEADERS, timeout=15)
        resp.raise_for_status()
    except requests.exceptions.RequestException as e:
        print(f'Ошибка загрузки страницы {page_num}: {e}')
        return results

    soup = BeautifulSoup(resp.text, 'html.parser')

    # Блоки новостей на странице списка
    # На сайте ITMO новости в <li class="weeklyevent"> или в секции news-list
    news_items = soup.find_all('li', class_='weeklyevent')

    # Резервный вариант: article или div с классом news
    if not news_items:
        news_items = soup.find_all('div', class_='news-item')
    if not news_items:
        news_items = soup.find_all('article')

    for item in news_items:
        link_tag = None
        date_text = ''

        # Ищем ссылку на новость
        h4 = item.find('h4')
        h3 = item.find('h3')
        h2 = item.find('h2')

        for heading in [h4, h3, h2]:
            if heading:
                link_tag = heading.find('a')
                if link_tag:
                    break

        if not link_tag:
            link_tag = item.find('a')

        if not link_tag:
            continue

        title = link_tag.get_text(strip=True)
        href = link_tag.get('href', '')

        if href.startswith('http'):
            full_url = href
        else:
            full_url = DOMAIN + href

        # Ищем дату в <p> или <time> внутри элемента
        p_tags = item.find_all('p')
        for p in p_tags:
            text = p.get_text(strip=True)
            # Дата в формате DD.MM.YYYY
            if re.match(r'\d{2}\.\d{2}\.\d{4}', text):
                date_text = text
                break

        if not date_text:
            time_tag = item.find('time')
            if time_tag:
                date_text = time_tag.get_text(strip=True)

        news_id = extract_id_from_url(full_url)

        if title and full_url and news_id:
            results.append({
                'id': news_id,
                'title': title,
                'date': date_text,
                'url': full_url
            })

    return results


print('Функции парсинга списка новостей готовы.')


In [ ]:
# Собираем список новостей со всех страниц
all_news = []
seen_ids = set()

print(f'Начинаю сбор списка новостей ({PAGES_TO_PARSE} страниц)...')

for page in range(1, PAGES_TO_PARSE + 1):
    print(f'  Парсинг страница {page}/{PAGES_TO_PARSE}...')
    page_news = parse_list_page(page)

    for item in page_news:
        if item['id'] not in seen_ids:
            all_news.append(item)
            seen_ids.add(item['id'])

    time.sleep(random.uniform(MIN_DELAY, MAX_DELAY))

print(f'\nВсего уникальных новостей собрано: {len(all_news)}')


In [ ]:
# Просмотр первых 5 записей
for item in all_news[:5]:
    print(f"ID: {item['id']}")
    print(f"Заголовок: {item['title'][:80]}")
    print(f"Дата: {item['date']}")
    print(f"URL: {item['url']}")
    print('-' * 60)


In [ ]:
# Сохранение общего списка в CSV
summary_fieldnames = ['id', 'title', 'date', 'url']

with open(SUMMARY_CSV, 'w', newline='', encoding='utf-8') as f:
    writer = csv.DictWriter(f, fieldnames=summary_fieldnames)
    writer.writeheader()
    writer.writerows(all_news)

print(f'Общий список сохранён в: {SUMMARY_CSV}')
print(f'Количество новостей: {len(all_news)}')


## 4. Парсинг подробных данных каждой новости

Для каждой новости из списка парсим:
- ID
- Название
- Дата размещения
- Количество просмотров
- Текст статьи (с учётом вариативности вёрстки)
- Теги


In [ ]:
def parse_news_page(url):
    """
    Парсит страницу отдельной новости.
    Возвращает словарь с подробными данными.
    Учитывает вариативность вёрстки на сайте ITMO.
    """
    result = {
        'id': extract_id_from_url(url),
        'title': '',
        'date': '',
        'views': '',
        'text': '',
        'tags': ''
    }

    try:
        resp = requests.get(url, headers=HEADERS, timeout=15)
        resp.raise_for_status()
    except requests.exceptions.RequestException as e:
        print(f'Ошибка загрузки {url}: {e}')
        return result

    soup = BeautifulSoup(resp.text, 'html.parser')

    # --- НАЗВАНИЕ ---
    # Вариант 1: h1 с классом
    title_tag = soup.find('h1', class_=re.compile(r'title|heading|news', re.I))
    # Вариант 2: любой h1
    if not title_tag:
        title_tag = soup.find('h1')
    if title_tag:
        result['title'] = title_tag.get_text(strip=True)

    # --- ДАТА ---
    # Вариант 1: div.time > time
    date_div = soup.find('div', class_='time')
    if date_div:
        time_tag = date_div.find('time')
        if time_tag:
            result['date'] = time_tag.get_text(strip=True)

    # Вариант 2: тег <time> в любом месте
    if not result['date']:
        time_tag = soup.find('time')
        if time_tag:
            # предпочтительно атрибут datetime
            dt = time_tag.get('datetime', '')
            result['date'] = dt if dt else time_tag.get_text(strip=True)

    # Вариант 3: элемент с классом date/published
    if not result['date']:
        for cls in ['date', 'published', 'post-date', 'news-date', 'article-date']:
            el = soup.find(class_=cls)
            if el:
                text = el.get_text(strip=True)
                if re.search(r'\d', text):
                    result['date'] = text
                    break

    # Вариант 4: span или p с датой в формате DD.MM.YYYY
    if not result['date']:
        for tag in soup.find_all(['span', 'p', 'div']):
            text = tag.get_text(strip=True)
            if re.match(r'^\d{2}\.\d{2}\.\d{4}$', text):
                result['date'] = text
                break

    # --- ПРОСМОТРЫ ---
    # Вариант 1: элемент с классом views/hits
    for cls in ['views', 'hits', 'views-count', 'counter', 'read-count', 'view-count']:
        el = soup.find(class_=cls)
        if el:
            text = el.get_text(strip=True)
            nums = re.findall(r'\d+', text)
            if nums:
                result['views'] = nums[0]
                break

    # Вариант 2: span/div содержащий слово 'просмотр' или 'views'
    if not result['views']:
        for tag in soup.find_all(['span', 'div', 'p']):
            text = tag.get_text(strip=True).lower()
            if ('просмотр' in text or 'views' in text or 'hits' in text) and re.search(r'\d', text):
                nums = re.findall(r'\d+', text)
                if nums:
                    result['views'] = nums[0]
                    break

    # --- ТЕКСТ СТАТЬИ ---
    # Вариант 1: div.news-text
    body = soup.find('div', class_='news-text')

    # Вариант 2: div с классом article__body или article-body
    if not body:
        body = soup.find('div', class_=re.compile(r'article.body|article-body|entry.content|post.content', re.I))

    # Вариант 3: article tag
    if not body:
        body = soup.find('article')

    # Вариант 4: div.content или div.text
    if not body:
        for cls in ['content', 'text', 'body', 'news-body', 'post-body', 'entry']:
            body = soup.find('div', class_=cls)
            if body:
                break

    # Вариант 5: main tag
    if not body:
        body = soup.find('main')

    if body:
        # Удаляем блокирующие элементы (навигация, реклама, поделиться)
        for unwanted in body.find_all(['script', 'style', 'nav', 'aside']):
            unwanted.decompose()
        for unwanted in body.find_all(class_=re.compile(r'share|social|sidebar|advert|banner|related', re.I)):
            unwanted.decompose()
        paragraphs = body.find_all('p')
        if paragraphs:
            text_parts = [p.get_text(strip=True) for p in paragraphs if p.get_text(strip=True)]
            result['text'] = ' '.join(text_parts)
        else:
            # Если нет тегов <p>, берём весь текст блока
            result['text'] = body.get_text(separator=' ', strip=True)

    # --- ТЕГИ ---
    tags_list = []

    # Вариант 1: div.tags или section.tags
    tags_block = soup.find(['div', 'section', 'ul'], class_=re.compile(r'tags|tag-list|keywords', re.I))
    if tags_block:
        for a in tags_block.find_all('a'):
            tag_text = a.get_text(strip=True)
            if tag_text:
                tags_list.append(tag_text)

    # Вариант 2: meta keywords
    if not tags_list:
        meta_kw = soup.find('meta', attrs={'name': 'keywords'})
        if meta_kw:
            content = meta_kw.get('content', '')
            if content:
                tags_list = [t.strip() for t in content.split(',') if t.strip()]

    # Вариант 3: ссылки с /tag/ в URL
    if not tags_list:
        for a in soup.find_all('a', href=re.compile(r'/tag/|/tags/|/topic/', re.I)):
            tag_text = a.get_text(strip=True)
            if tag_text and tag_text not in tags_list:
                tags_list.append(tag_text)

    result['tags'] = '|'.join(tags_list)

    return result


print('Функция парсинга отдельной новости готова.')


In [ ]:
# Тестовый запуск на первой новости
if all_news:
    test_url = all_news[0]['url']
    print(f'Тест: парсинг {test_url}')
    test_result = parse_news_page(test_url)
    print(f"ID       : {test_result['id']}")
    print(f"Заголовок: {test_result['title'][:80]}")
    print(f"Дата     : {test_result['date']}")
    print(f"Просмотры: {test_result['views']}")
    print(f"Текст    : {test_result['text'][:200]}...")
    print(f"Теги     : {test_result['tags']}")


## 5. Сбор и сохранение подробных данных всех новостей


In [ ]:
# Имена полей для CSV с подробными данными
detail_fieldnames = ['id', 'title', 'date', 'views', 'text', 'tags']

# Файл с объединёнными данными всех новостей
all_details_csv = os.path.join(NEWS_CONTENT_DIR, 'all_news_details.csv')

failed_urls = []
processed_count = 0

print(f'Начинаю сбор подробных данных ({len(all_news)} новостей)...')
print(f'Отдельные CSV-файлы будут в папке: {NEWS_CONTENT_DIR}/')
print()

# Открываем общий CSV для записей
with open(all_details_csv, 'w', newline='', encoding='utf-8') as all_f:
    all_writer = csv.DictWriter(all_f, fieldnames=detail_fieldnames)
    all_writer.writeheader()

    for i, news_item in enumerate(all_news):
        news_id = news_item['id']
        url = news_item['url']

        details = parse_news_page(url)

        # Если title не удалось спарсить со страницы - берём из списка
        if not details['title']:
            details['title'] = news_item['title']
        if not details['date']:
            details['date'] = news_item['date']

        # Записываем в общий файл
        all_writer.writerow(details)

        # Записываем в отдельный файл для каждой новости
        single_csv = os.path.join(NEWS_CONTENT_DIR, f'news_{news_id}.csv')
        with open(single_csv, 'w', newline='', encoding='utf-8') as sf:
            single_writer = csv.DictWriter(sf, fieldnames=detail_fieldnames)
            single_writer.writeheader()
            single_writer.writerow(details)

        processed_count += 1

        if processed_count % 5 == 0:
            print(f'  Обработано {processed_count}/{len(all_news)} новостей...')

        # Задержка между запросами
        time.sleep(random.uniform(MIN_DELAY, MAX_DELAY))

print()
print(f'Сбор данных завершён!')
print(f'Всего обработано: {processed_count} новостей')
print(f'Общий CSV: {all_details_csv}')
print(f'Отдельные файлы: {NEWS_CONTENT_DIR}/news_<ID>.csv')


## 6. Проверка результатов


In [ ]:
# Проверка структуры папки
print('=== Структура файлов ===')
print(f'{SUMMARY_CSV}  (общий список новостей)')
print(f'{NEWS_CONTENT_DIR}/')

files_in_dir = os.listdir(NEWS_CONTENT_DIR)
csv_files = [f for f in files_in_dir if f.endswith('.csv')]
print(f'  all_news_details.csv  (все подробные данные)')
print(f'  news_XXXXX.csv x{len(csv_files) - 1}  (отдельные файлы новостей)')
print(f'  Всего файлов в папке: {len(csv_files)}')


In [ ]:
# Проверка содержимого общего файла
print('=== Первые 3 строки из news_summary.csv ===')
with open(SUMMARY_CSV, 'r', encoding='utf-8') as f:
    reader = csv.DictReader(f)
    for i, row in enumerate(reader):
        if i >= 3:
            break
        print(f"  id={row['id']}, date={row['date']}, title={row['title'][:50]}...")

print()
print('=== Первые 3 строки из all_news_details.csv ===')
with open(all_details_csv, 'r', encoding='utf-8') as f:
    reader = csv.DictReader(f)
    for i, row in enumerate(reader):
        if i >= 3:
            break
        print(f"  id={row['id']}, views={row['views']}, tags={row['tags'][:40]}")
        print(f"  текст (первые 100 символов): {row['text'][:100]}")
        print()


In [ ]:
# Статистика парсинга
total_with_text = 0
total_with_tags = 0
total_with_views = 0
total_rows = 0

with open(all_details_csv, 'r', encoding='utf-8') as f:
    reader = csv.DictReader(f)
    for row in reader:
        total_rows += 1
        if row['text']:
            total_with_text += 1
        if row['tags']:
            total_with_tags += 1
        if row['views']:
            total_with_views += 1

print('=== Статистика парсинга ===')
print(f'Всего новостей в CSV       : {total_rows}')
print(f'С текстом                  : {total_with_text} ({100*total_with_text//max(total_rows,1)}%)')
print(f'С тегами                   : {total_with_tags} ({100*total_with_tags//max(total_rows,1)}%)')
print(f'С количеством просмотров   : {total_with_views} ({100*total_with_views//max(total_rows,1)}%)')


## 7. Скачивание файлов из Colab

Скачаем сохранённые CSV-файлы на локальную машину.


In [ ]:
from google.colab import files

# Скачивание общего списка
print('Скачивание news_summary.csv...')
files.download(SUMMARY_CSV)

# Скачивание файла с подробными данными
print('Скачивание all_news_details.csv...')
files.download(all_details_csv)


In [ ]:
# Архивируем папку news_content и скачиваем
import shutil

archive_name = 'news_content_archive'
shutil.make_archive(archive_name, 'zip', NEWS_CONTENT_DIR)

print(f'Архив создан: {archive_name}.zip')
files.download(f'{archive_name}.zip')
